<a href="https://colab.research.google.com/github/Eleonwra/elcardiocc-baseline-ner/blob/main/results/test_predictions_mbert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install datasets
%pip install seqeval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 14.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system ==

In [2]:
from transformers import BertForTokenClassification
from transformers import BertTokenizerFast
from torch.utils.data import Dataset, DataLoader
from datasets import Dataset
from itertools import product
from sklearn.metrics import classification_report, accuracy_score, f1_score
from seqeval.metrics import classification_report as seqeval_classification_report
from torch.optim import AdamW
from torch.optim.lr_scheduler import StepLR
from torch.nn import CrossEntropyLoss
import numpy as np
import warnings
import os

warnings.filterwarnings("ignore")

In [53]:
from transformers import AutoModelForTokenClassification, AutoTokenizer
from google.colab import drive
import pickle
import shutil
import os
drive.mount('/content/drive')
drive_model_path = "/content/drive/MyDrive/Baseline_ELCardioCC_data/model_training/best_configuration_mbert"
model = AutoModelForTokenClassification.from_pretrained(drive_model_path)
tokenizer = AutoTokenizer.from_pretrained(drive_model_path)
with open('/content/drive/MyDrive/Baseline_ELCardioCC_data/model_training/mBERT_tokenized_test_documents_trun.pkl', 'rb') as file:
  test_data = pickle.load(file)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [51]:
import json
data = []
with open('test_dataset.jsonl', 'r') as f:
    for line in f:
        data.append(json.loads(line))

In [52]:
texts = []
for entry in data:
    patient_id = entry['id']
    text = entry['text']
    texts.append({
            'patient_id': patient_id,
            'text': text
        })

In [54]:
import re
import torch
from collections import Counter
from transformers import BertTokenizerFast, BertForTokenClassification

def get_ner_predictions(model, document, label_map, current_text, device='cpu'):
    model.to(device)
    model.eval()

    tokenizer = AutoTokenizer.from_pretrained(drive_model_path)

    all_filtered_tokens = []
    all_filtered_preds = []
    all_input_ids = []
    all_offsets = []

    for i in range(len(document['input_ids'])):
        input_ids = torch.tensor(document['input_ids'][i]).unsqueeze(0).to(device)
        attention_mask = torch.tensor(document['attention_mask'][i]).unsqueeze(0).to(device)
        token_type_ids = (
            torch.tensor(document['token_type_ids'][i]).unsqueeze(0).to(device)
            if 'token_type_ids' in document
            else None
        )

        with torch.no_grad():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )
            logits = outputs.logits
            predictions = torch.argmax(logits, dim=-1).squeeze().cpu().numpy()

            input_ids = input_ids.squeeze().cpu().numpy()

            tokens = tokenizer.convert_ids_to_tokens(input_ids)
            pred_labels = [label_map[pred] for pred in predictions]

            for token, label in zip(tokens, pred_labels):
                if token not in tokenizer.all_special_tokens:
                    all_filtered_tokens.append(token)
                    all_filtered_preds.append(label)
                    input_id = tokenizer.convert_tokens_to_ids(token)
                    all_input_ids.append(input_id)
    all_input_ids = torch.tensor(all_input_ids)
    encoding = tokenizer.decode(all_input_ids.squeeze(), skip_special_tokens=False)

    #text = tokenizer.convert_tokens_to_string(all_filtered_tokens)
    entities = extract_words_from_subwords(all_filtered_tokens, all_filtered_preds, current_text)
    final_entities = group_consecutive_entities(entities)

    return {
        "text": current_text,
        "tokens": all_filtered_tokens,
        "labels": all_filtered_preds,
        "entities": final_entities
    }
def extract_words_from_subwords(tokens, labels, text):
    entities = []
    current_tokens = []
    current_labels = []
    search_start = 0

    for i, (token, label) in enumerate(zip(tokens, labels)):
        if not token.startswith('##'):
            if current_tokens:
                majority_label = Counter(current_labels).most_common(1)[0][0]
                entity_text = ''.join(current_tokens)
                #match = re.search(re.escape(entity_text), text, search_start)
                match = re.search(re.escape(entity_text), text[search_start:], re.ASCII)
                if match:
                    start, end = match.span()
                    start += search_start
                    end += search_start
                    entities.append({
                        "mention": entity_text,
                        "label": majority_label,
                        "start": start,
                        "end": end
                    })
                    search_start = end
                current_tokens = []
                current_labels = []

            current_tokens = [token]
            current_labels = [label]

        else:
            current_tokens.append(token[2:])
            current_labels.append(label)
    if current_tokens:
        majority_label = Counter(current_labels).most_common(1)[0][0]
        entity_text = ''.join(current_tokens)

        #match = re.search(re.escape(entity_text), text, search_start)
        match = re.search(re.escape(entity_text), text[search_start:], re.ASCII)
        if match:
            start, end = match.span()
            start += search_start
            end += search_start
            entities.append({
                "mention": entity_text,
                "label": majority_label,
                "start": start,
                "end": end
            })
            search_start = end

    return entities
def group_consecutive_entities(entities):
    grouped_entities = []
    current_group = []

    for i, entity in enumerate(entities):
        if entity['label'] != 'O':
            if current_group:
                last_entity = current_group[-1]
                if (last_entity['end'] == entity['start']-1) and (
                        (last_entity['label'] == entity['label']) or
                        (last_entity['label'] == 'B-ENTITY' and entity['label'] == 'I-ENTITY')):
                    current_group.append(entity)
                else:
                    grouped_entities.append(current_group)
                    current_group = [entity]
            else:
                current_group = [entity]
        else:
            if current_group:
                grouped_entities.append(current_group)
                current_group = []
    if current_group:
        grouped_entities.append(current_group)

    final_entities = []
    for group in grouped_entities:
        merged_entity_text = ' '.join([entity['mention'] for entity in group])
        majority_label = Counter([entity['label'] for entity in group]).most_common(1)[0][0]

        start = group[0]['start']
        end = group[-1]['end']

        final_entities.append({
            "mention": merged_entity_text,
            "label": majority_label,
            "start": start,
            "end": end
        })

    return final_entities

In [ ]:
label_map = {
    0: "O",
    1: "B-ENTITY",
    2: "I-ENTITY"
}
final_results = []

for i, document in enumerate(test_data['test']):
    current_text = texts[i]['text']
    doc_results = get_ner_predictions(AutoModelForTokenClassification.from_pretrained(drive_model_path), document, label_map, current_text)
    final_results.append({
        "text": doc_results['text'],
        "tokens": doc_results['tokens'],
        "labels": doc_results['labels'],
        "entities": doc_results['entities']
    })

In [ ]:
import json

def create_json_from_results(results):
    json_data = {
        "system_name": "ELCardioCC_ner_baseline",
        "submission": []
    }

    for i, result in enumerate(results):
        doc_annotations = []
        for mentions in result['entities']:
            mention = mentions['mention']
            start = mentions['start']
            end = mentions['end']

            annotation = {
                "start": start,
                "end": end,
                "mention": mention,
                "code": ""
            }
            doc_annotations.append(annotation)


        json_data["submission"].append({
            "id": texts[i]['patient_id'],
            "annotations": doc_annotations
        })
    with open('/content/drive/MyDrive/Baseline_ELCardioCC_data/model_training/NER.json', 'w', encoding='utf-8') as f:
        json.dump(json_data, f, ensure_ascii=False, indent=4)

In [ ]:
create_json_from_results(final_results)